In [ ]:
!pip install langchain_community

In [1]:
from langchain_community.document_loaders import PyPDFLoader

# Documento de exemplo a ser inserido. Pode ser inserido qualquer outro pdf.
pdf_path = 'ArtigoIAEducacao.pdf'

loader = PyPDFLoader(pdf_path)
pages = loader.load_and_split()

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Divide o texto em chunks. O chunk_size pode ser alterado para melhor se adequar ao conteúdo.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len,
)

texts = text_splitter.split_documents(pages)

# Imprime o primeiro chunk
print(texts[0])

page_content='1https://doi.org/10.1590/2318-0889202436e2410839
TransInformação I Campinas I v. 36 I e2410839 I 2024
Inteligência Artificial Generativa e 
ChatGPT: uma investigação sobre 
seu potencial na Educação
Generative Artificial Intelligence and 
ChatGPT: an investigation into 
perception in Education
Cleosanice Barbosa Lima1      , Agostinho Serrano1     
1 Universidade Luterana do Brasil, Pró-Reitoria Acadêmica, Programa de Pós-Graduação em Ensino de Ciências e 
Matemática. Canoas, RS, Brasil. Correspondência para/Correspondence to: C. B. LIMA. E-mail: <cblimacleo@gmail.com>.
 Como citar este artigo/How to cite this article: Lima, C. B.; Serrano, A. Inteligência Artificial Generativa e 
ChatGPT : uma investigação sobre seu potencial na Educação. Transinformação, v. 36, e2410839, 2024. https://doi.
org/10.1590/2318-0889202436e2410839 
Resumo
Este texto apresenta uma revisão de literatura, explorando os impactos do chatbot ChatGPT' metadata={'producer': 'Adobe PDF Library 15.0', 

In [3]:
# Imprime o numero de chunks criados. Opciomal.
print (f"Quantidade de chunks: {len(texts)} ")

Quantidade de chunks: 51 


In [ ]:
# Pode ser instalado qualquer uma das duas opções (CPU ou GPU) a depender do que estiver disponível.
!pip install faiss-cpu
#pip install faiss-gpu

In [4]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Cria o banco de dados com os embeddings
db = FAISS.from_documents(texts,  OllamaEmbeddings(model="mxbai-embed-large"))

C:\Users\alyne\AppData\Local\Temp\ipykernel_27088\1269879013.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  db = FAISS.from_documents(texts,  OllamaEmbeddings(model="mxbai-embed-large"))


In [5]:
# Trecho opcional e serve para testar o código e verificar os embeddings.
query = "De acordo com o texto, o que é inteligencia Artificial?"
docs = db.similarity_search(query)
print(docs[0].page_content)

Artificial Generativa − o ChatGPT ; Inteligência Artificial com foco no ChatGPT , possibilidades,


In [6]:
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA

# Determina o modelo de busca
model = OllamaLLM(model="llama3.2:latest")

# Recuperando o retriever com 5 documentos
retriever = db.as_retriever(search_kwargs={"k": 5})
#retriever = db.as_retriever()

# Criando um RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(llm=model, retriever=retriever, chain_type="stuff")

In [7]:
# Exemplo de uso do RetrievalQA chain
# É possivel substituir o query_retriever por outros questionamentos.
query_retrieval = "Quais aspectos éticos o artigo menciona sobre uso do chatgpt na educação?"
response = qa_chain.invoke(query_retrieval)
print("QA Response:", response)

QA Response: {'query': 'Quais aspectos éticos o artigo menciona sobre uso do chatgpt na educação?', 'result': 'O artigo menciona várias questões éticas relacionadas ao uso do ChatGPT na educação, incluindo:\n\n*   Estímulo ao plágio: O uso de ChatGPT pode incentivar os alunos a copiar e colar conteúdo sem entender seu significado, o que pode ser prejudicial à compreensão e ao desenvolvimento de habilidades críticas.\n*   Inibição da criatividade dos alunos: A automação do trabalho criativo com ChatGPT pode reduzir as oportunidades para os alunos expressarem suas ideias e pensamentos originais.\n*   Falta de transparência: Os professores devem ser transparentes sobre o uso de ChatGPT como ferramenta de apoio, evitando ocultar a sua presença nas atividades de aprendizagem.\n*   Avaliação de conteúdo gerado por ChatGPT: O artigo destaca a necessidade de desenvolver critérios e métodos para avaliar o conteúdo produzido pelo ChatGPT, garantindo que seja justo e equitativo.\n\nEssas questões